In [1]:
%pip install -U -q python-dotenv pypdf langchain-core langchain-community langchain-text-splitters langchain-openai pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from the notebook's folder

for name in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    value = os.getenv(name)
    print(name, "loaded" if value else "MISSING", f"(ends in ...{value[-4:]})" if value else "")


OPENAI_API_KEY loaded (ends in ...z40A)
PINECONE_API_KEY loaded (ends in ...9Aoq)


In [9]:
%pip install langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("data")
PODCAST_PATH = DATA_DIR / "podcast.txt"
PDF_PATH = DATA_DIR / "eu_ai_act.pdf"

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the lab files in a 'data' folder next to this notebook.")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()  # 1 Document (whole transcript)
pdf_docs = PyPDFLoader(str(PDF_PATH)).load()                           # 1 Document per PDF page

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("\nRaw metadata from the PDF loader:", pdf_docs[0].metadata)

Podcast: 1 document, 16,414 characters
EU AI Act: 144 pages

Raw metadata from the PDF loader: {'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'creator': 'Servigistics Arbortext Publishing Engine', 'creationdate': '2024-07-11T14:47:17+02:00', 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'subject': 'I Legislative acts', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'source': 'data/eu_ai_act.pdf', 'total_pages': 144, 'page': 0, 'page_label': '1'}


In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

import re
ARTICLE_RE = re.compile(r"^Article\s+(\d+)\s*$", re.MULTILINE)   # heading lines only, not "see Article 5(1)"

def chunk_with_metadata(docs, source_type, source_name):
    chunks = splitter.split_documents(docs)
    current_section = "recital"      # the Act starts with recitals
    current_article = None
    for i, chunk in enumerate(chunks):
        text = chunk.page_content
        metadata = {
            "source_type": source_type,
            "source_name": source_name,
            "chunk_id": f"{source_type}-{i}",
        }
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1

        if source_type == "regulation":
            # section switches at the "CHAPTER I" and "ANNEX I" headings (p.44 and p.124)
            if re.search(r"^CHAPTER I\s*$", text, re.MULTILINE):
                current_section = "article"
            if re.search(r"^ANNEX I\s*$", text, re.MULTILINE):
                current_section = "annex"

            # a chunk belongs to the article it continues + any article heading inside it
            starts_with_heading = re.match(r"Article\s+\d+\s*\n", text)
            articles = [current_article] if current_article and not starts_with_heading else []
            for num in ARTICLE_RE.findall(text):
                articles.append(f"Article {num}")
                current_article = f"Article {num}"

            metadata["section"] = current_section
            if current_section == "article" and articles:          # never store None in Pinecone
                metadata["articles"] = sorted(set(articles), key=lambda a: int(a.split()[1]))

        chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

print([c.metadata for c in act_chunks if c.metadata.get("page") == 115][2])

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784
{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-665', 'page': 115, 'section': 'article', 'articles': ['Article 99']}


In [26]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector = embeddings.embed_query(all_chunks[0].page_content)
print(len(vector))


1536


In [27]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "lab-rerankers"

# Create the index only if it doesn't exist yet
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,            # must match text-embedding-3-small
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

# Upload only if the index is empty -> safe to re-run this cell
stats = pc.Index(index_name).describe_index_stats()
if stats.total_vector_count == 0:
    vectorstore = PineconeVectorStore.from_documents(
        documents=all_chunks, embedding=embeddings, index_name=index_name
    )
    print(f"Uploaded {len(all_chunks)} chunks")
else:
    vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)
    print(f"Index already has {stats.total_vector_count} vectors, connected without re-uploading")

Index already has 784 vectors, connected without re-uploading


In [14]:
# PROVIDED: fixed test queries. Use the SAME queries in every step so you can compare fairly.
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

# PROVIDED: helper to print results in the same format in every step
def show_results(query, results, label="Results"):
    """results: list of (Document, score) pairs"""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m.get("chunk_id")
        print(f"#{rank}  score={score:.3f}  [{m.get('source_type')} | {where}]  {doc.page_content[:120]!r}")

In [16]:
# YOUR TURN: baseline retrieval
def baseline_search(query, k=5):
    # TODO: return vectorstore.similarity_search_with_score(...)
    pass

baseline_results = {}
for q in TEST_QUERIES:
    # TODO: call baseline_search, store the result in baseline_results[q], and print it with show_results
    pass

In [17]:
def baseline_search(query, k=5):
    return vectorstore.similarity_search_with_score(query, k=k)

baseline_results = {}
for q in TEST_QUERIES:
    baseline_results[q] = baseline_search(q)
    show_results(q, baseline_results[q], label="Baseline")


=== Baseline | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.682  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'
#4  score=0.673  [regulation | p.45.0]  '(b) prohibitions of certain AI practices;\n(c) specific requirements for high-risk AI systems and obligations for operato'
#5  score=0.660  [regulation | p.3.0]  'operators involved in the AI value chain under this Regulation should apply without prejudice to national law, in \ncompl'

=== Baseline | Query: What are the seven requirements for trustworthy AI?
#1  score=0.674  [podcast | p

# What the baseline shows

# Q4 (fines) is my best “first match ≠ right match” case.

# #1 is p.51, Article 4 – AI literacy. That has nothing to do with fines.
# #2 is p.118, which is the end of Article 101. Those are fines for providers of general-purpose AI models, so it’s the wrong article.
# The right answer is on p.115, Article 99(3): fines of up to €35,000,000 or 7% of total worldwide annual turnover. It isn’t in my top 5 at all.

# Q1 (prohibited practices) is half-right. The top chunk starts with Article 4, but Article 5 “Prohibited AI practices” begins further down the same page. The answer is probably at the end of the chunk, after an unrelated heading. The other results are mostly recitals, which explain the background but are not the binding rule.

# Q2 and Q3 work well, because only the podcast covers those topics.

In [18]:
%pip install -q sentence-transformers
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def crossencoder_rerank(query, k_initial=10, k_final=5):
    candidates = baseline_search(query, k=k_initial)          # wider net
    docs = [doc for doc, _ in candidates]
    scores = reranker.predict([(query, d.page_content) for d in docs])
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [(doc, float(score)) for doc, score in ranked[:k_final]]

Note: you may need to restart the kernel to use updated packages.


W1027 14:28:42.626000 76250 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 17647.14it/s]


In [19]:
reranked_results = {}
for q in TEST_QUERIES:
    reranked_results[q] = crossencoder_rerank(q)
    show_results(q, baseline_results[q][:3], label="Baseline top 3")
    show_results(q, reranked_results[q][:3], label="Reranked top 3")


=== Baseline top 3 | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.682  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'

=== Reranked top 3 | Query: What AI practices are prohibited under the EU AI Act?
#1  score=6.939  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#2  score=6.909  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'
#3  score=6.685

# Reranker made some results better and some worse

# Query	What changed	Better or worse?
# Q4 Fines	p.115 moved from #3 → #1. That chunk contains Article 99(3): “fines of up to EUR 35 000 000 or … 7 % of its total worldwide annual turnover”	Clearly better. This is your headline case
# Q2 Seven requirements	Recital 27 (p.8) moved to #1, above the podcast intro	Better. Recital 27 lists all seven requirements from the 2019 guidelines. The podcast chunk was only the intro
# Q3 HITL vs HOTL	Same #1 (podcast-8)	No change. The baseline was already good
# Q1 Prohibited	Recitals (45) and p.9 moved up; the p.51 chunk with Article 5 dropped to #3	Sideways or worse. Recitals explain the bans, but Article 5 is the ban
# Q5 Social scoring	p.51 Article 5(1)(c), the binding ban on social scoring, fell out of the top 3 and a podcast chunk took its place	Worse. All the scores are negative too, which means the model wasn’t confident

# The reranker finds chunks that talk about the question well, but it can’t tell a binding article from an explanatory recital. Metadata is what solves that, and it’s the next step.

In [20]:
# Metadata filtering

q = "Is social scoring of citizens allowed?"
show_results(q, vectorstore.similarity_search_with_score(q, k=5, filter={"source_type": {"$eq": "regulation"}}), "Regulation only")
show_results(q, vectorstore.similarity_search_with_score(q, k=5, filter={"source_type": {"$eq": "podcast"}}), "Podcast only")


=== Regulation only | Query: Is social scoring of citizens allowed?
#1  score=0.588  [regulation | p.9.0]  'multiple data points related to their social behaviour in multiple contexts or known, inferred or predicted personal \nor'
#2  score=0.497  [regulation | p.9.0]  'face or fingerprint, to deduce or infer an individuals’ political opinions, trade union membership, religious or \nphilos'
#3  score=0.497  [regulation | p.51.0]  '(c) the placing on the market, the putting into service or the use of AI systems for the evaluation or classification of'
#4  score=0.421  [regulation | p.16.0]  'should be granted, denied, reduced, revoked or reclaimed by authorities, including whether beneficiaries are \nlegitimate'
#5  score=0.412  [regulation | p.127.0]  '(a) AI systems intended to be used by public authorities or on behalf of public authorities to evaluate the eligibility '

=== Podcast only | Query: Is social scoring of citizens allowed?
#1  score=0.485  [podcast | podcast-16]  'learni

In [21]:
filter={"section": {"$eq": "article"}}

In [31]:
import re
ARTICLE_RE = re.compile(r"^Article\s+(\d+)\s*$", re.MULTILINE)   # heading lines only, not "see Article 5(1)"

def chunk_with_metadata(docs, source_type, source_name):
    chunks = splitter.split_documents(docs)
    current_section = "recital"      # the Act starts with recitals
    current_article = None
    for i, chunk in enumerate(chunks):
        text = chunk.page_content
        metadata = {
            "source_type": source_type,
            "source_name": source_name,
            "chunk_id": f"{source_type}-{i}",
        }
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1

        if source_type == "regulation":
            # section switches at the "CHAPTER I" and "ANNEX I" headings (p.44 and p.124)
            if re.search(r"^CHAPTER I\s*$", text, re.MULTILINE):
                current_section = "article"
            if re.search(r"^ANNEX I\s*$", text, re.MULTILINE):
                current_section = "annex"

            # a chunk belongs to the article it continues + any article heading inside it
            starts_with_heading = re.match(r"Article\s+\d+\s*\n", text)
            articles = [current_article] if current_article and not starts_with_heading else []
            for num in ARTICLE_RE.findall(text):
                articles.append(f"Article {num}")
                current_article = f"Article {num}"

            metadata["section"] = current_section
            if current_section == "article" and articles:          # never store None in Pinecone
                metadata["articles"] = sorted(set(articles), key=lambda a: int(a.split()[1]))

        chunk.metadata = metadata
    return chunks

In [23]:
q = "Is social scoring of citizens allowed?"
show_results(q, vectorstore.similarity_search_with_score(q, k=5, filter={"section": {"$eq": "article"}}), "Articles only")

q = "What fines apply for using a prohibited AI practice?"
show_results(q, vectorstore.similarity_search_with_score(q, k=5, filter={"articles": {"$in": ["Article 99"]}}), "Article 99 only")


=== Articles only | Query: Is social scoring of citizens allowed?

=== Article 99 only | Query: What fines apply for using a prohibited AI practice?


In [24]:
# 1) Does the local data have the new fields?
print("Local chunk p.115:", [c.metadata for c in act_chunks if c.metadata.get("page") == 115][:2])

# 2) What does Pinecone actually have stored?
print("Vectors in index:", pc.Index(index_name).describe_index_stats().total_vector_count)
doc, score = vectorstore.similarity_search_with_score("administrative fines Article 99", k=1)[0]
print("Stored metadata:", doc.metadata)

Local chunk p.115: [{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-663', 'page': 115, 'text': '5. As soon as it adopts a delegated act, the Commission shall notify it simultaneously to the European Parliament and to \nthe Council.\n6. Any delegated act adopted pursuant to Article 6(6) or (7), Article 7(1) or (3), Article 11(3), Article 43(5) or (6), \nArticle 47(5), Article 51(3), Article 52(4) or Article 53(5) or (6) shall enter into force only if no objection has been \nexpressed by either the European Parliament or the Council within a period of three months of notification of that act to \nthe European Parliament and the Council or if, before the expiry of that period, the European Parliament and the Council \nhave both informed the Commission that they will not object. That period shall be extended by three months at the \ninitiative of the European Parliament or of the Council.\nArticle 98\nCommittee procedure\n1. The Commission

In [29]:
pc.delete_index(index_name)

In [33]:
import time

if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name, dimension=1536, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))

while not pc.describe_index(index_name).status["ready"]:   # wait until Pinecone says ready
    print("waiting for index...")
    time.sleep(5)

if pc.Index(index_name).describe_index_stats().total_vector_count == 0:
    vectorstore = PineconeVectorStore.from_documents(all_chunks, embedding=embeddings, index_name=index_name)
    print(f"Uploaded {len(all_chunks)} chunks")
    time.sleep(10)                                            # let the new vectors become searchable
else:
    vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)
    print("Connected to existing index")

print("Vectors in index:", pc.Index(index_name).describe_index_stats().total_vector_count)

Uploaded 784 chunks
Vectors in index: 784


In [34]:
q = "Is social scoring of citizens allowed?"
show_results(q, vectorstore.similarity_search_with_score(q, k=5, filter={"section": {"$eq": "article"}}), "Articles only")

q = "What fines apply for using a prohibited AI practice?"
show_results(q, vectorstore.similarity_search_with_score(q, k=5, filter={"articles": {"$in": ["Article 99"]}}), "Article 99 only")


=== Articles only | Query: Is social scoring of citizens allowed?
#1  score=0.497  [regulation | p.51.0]  '(c) the placing on the market, the putting into service or the use of AI systems for the evaluation or classification of'
#2  score=0.390  [regulation | p.51.0]  'assessments of natural persons in order to assess or predict the risk of a natural person committing a criminal offence,'
#3  score=0.374  [regulation | p.52.0]  '(g) the placing on the market, the putting into service for this specific purpose, or the use of biometric categorisatio'
#4  score=0.367  [regulation | p.58.0]  '5. To the extent that it is strictly necessary for the purpose of ensuring bias detection and correction in relation to '
#5  score=0.358  [regulation | p.91.0]  'purpose of developing, training and testing certain AI systems in the sandbox when all of the following conditions are m'

=== Article 99 only | Query: What fines apply for using a prohibited AI practice?
#1  score=0.613  [regulation | p.11

# RAG pipeline 

In [35]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_context(results):
    """Put a citation label in front of every chunk so the LLM can cite it."""
    blocks = []
    for doc, _ in results:
        m = doc.metadata
        where = f"p.{int(m['page'])}" if "page" in m else m["chunk_id"]
        arts = f", {', '.join(m['articles'])}" if m.get("articles") else ""
        blocks.append(f"[{m['source_name']}, {where}{arts}]\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)

rag_prompt = ChatPromptTemplate.from_template(
    """You answer questions about trustworthy AI and the EU AI Act.
Use ONLY the context below. After every claim, cite the label of the chunk it came from,
exactly as written, e.g. [EU AI Act (Reg. 2024/1689), p.115, Article 99].
If the context does not contain the answer, reply exactly: "I don't know based on the sources."
Do not use outside knowledge.

Context:
{context}

Question: {question}

Answer:"""
)

chain = rag_prompt | llm | StrOutputParser()

def answer(query, retrieve_fn=baseline_search):    # <- swap retrieve_fn to switch methods
    results = retrieve_fn(query)
    response = chain.invoke({"context": format_context(results), "question": query})
    return response, results

In [36]:
q = "What fines apply for using a prohibited AI practice?"

ans_base, res_base = answer(q, baseline_search)
ans_rr,   res_rr   = answer(q, crossencoder_rerank)

print("BASELINE:\n", ans_base)
print("\nRERANKED:\n", ans_rr)
show_results(q, res_rr, "Chunks used by reranked answer")

BASELINE:
 Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act (Reg. 2024/1689), p.115, Article 99].

RERANKED:
 Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act (Reg. 2024/1689), p.115, Article 99].

=== Chunks used by reranked answer | Query: What fines apply for using a prohibited AI practice?
#1  score=4.259  [regulation | p.115.0]  'including start-ups, and their economic viability.\n2. The Member States shall, without delay and at the latest by the da'
#2  score=4.098  [regulation | p.117.0]  '

In [ ]:
# I dont know instruction test !!!

print(answer("How does the EU AI Act regulate quantum computing?", crossencoder_rerank)[0])

I don't know based on the sources.


# I predicted the baseline would fail here, and I was wrong. The reason is a useful finding: p.115 was #3 in the baseline’s top 5, so it was still in the context, and the LLM reads all 5 chunks no matter what order they’re in. So for this query, reranking changed the order but not the answer. Step 7 asks you exactly this (“Did reranking change the answer, or only the order?”), and you now have direct evidence for it.

# Reranking matters when the right chunk is at position 6–10, so it only reaches the context because of reranking. It also matters when you send fewer chunks to the LLM to save cost. If you want to show that, run answer(q, lambda q: baseline_search(q, k=2)). The €35M chunk then drops out of the context.

# There’s also a trace point for your proof. The answer cites Article 99, but the chunk text itself doesn’t contain the heading “Article 99”, because it starts mid-article. That label comes from your regex metadata. If the regex had tagged the chunk wrong, the citation would look just as convincing and still be wrong. That’s a strong candidate for “plausible but unsupported”.

In [38]:
import time

eval_runs = {}
for q in TEST_QUERIES:
    for name, fn in [("Baseline", baseline_search), ("Reranked", crossencoder_rerank)]:
        t0 = time.time()
        ans, res = answer(q, fn)
        secs = time.time() - t0
        eval_runs[(q, name)] = (ans, res, secs)
        print(f"\n{'='*90}\n{name} | {q}  ({secs:.1f}s)\n{ans}")
        show_results(q, res[:3], f"{name} top 3")


Baseline | What AI practices are prohibited under the EU AI Act?  (6.5s)
The following AI practices are prohibited under the EU AI Act: 

1. The placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision [EU AI Act (Reg. 2024/1689), p.51, Article 5]. 

2. Practices that lead to economic or financial harms to consumers through unfair commercial practices, irrespective of whether they are implemented through AI systems or otherwise [EU AI Act (Reg. 2024/1689), p.9].

=== Baseline top 3 | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, 